
# Chapter 26 Robotics — Revised Teaching Notebook

This notebook is a **student-facing, Colab-ready introduction** to the main ideas from Chapter 26 on robotics in *Artificial Intelligence: A Modern Approach*.

## What this notebook emphasizes

1. **Core ideas first**: what a bachelor-level CS student should actually remember.
2. **Practical demos**: small runnable examples rather than prose only.
3. **Exercises**: short questions after major sections.
4. **Lecture usability**: suitable for a 45–90 minute class or self-study session.

## Main learning outcomes

By the end of the notebook, you should be able to explain:

- why robotics is harder than many classical AI problems;
- the difference between **perception**, **planning**, and **control**;
- what **localization** means and why probabilistic methods are useful;
- what **configuration space** is and why it matters;
- the difference between a **path**, a **trajectory**, and a **control policy**;
- why uncertainty, real-world physics, and humans make robotics especially difficult.



## Lecture map

This notebook follows this structure:

1. Robots as physical AI agents  
2. Problem formulation: perception, planning, control  
3. Demo 1: particle-filter localization  
4. Configuration space and motion planning  
5. Demo 2: A* motion planning on a grid  
6. Trajectory tracking and feedback control  
7. Demo 3: P vs PD control  
8. Uncertainty, reinforcement learning, and human–robot interaction  
9. Exercises and recap

---

## A note on scope

This notebook does **not** try to reproduce the whole chapter in compressed textbook form. It instead focuses on the concepts most worth retaining after a first pass.


In [ ]:

# Colab-ready setup: only standard scientific Python packages.
import math
import heapq
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
print("Setup complete.")



## 1. Robots as physical AI agents

A robot is not just software. It is an **agent with a body**.

### Core idea

A robot senses the world, reasons about it, and acts on it through physical effectors.

### The basic loop

- **Sensors**: cameras, lidar, radar, microphones, encoders, force sensors
- **State estimation / perception**: what is around me? where am I?
- **Planning / decision making**: what should I do next?
- **Control**: what motor commands should I send right now?
- **Effectors / actuators**: wheels, legs, grippers, joints, propellers

### Why robotics is hard

Robotics is difficult because the world is:

- **partially observable** — sensors are noisy and incomplete;
- **stochastic** — wheels slip, people move unpredictably, objects shift;
- **continuous** — positions, angles, and velocities are usually real-valued;
- **high-dimensional** — even a simple robot arm can have many degrees of freedom;
- **safety-critical** — errors damage objects, environments, or people.

### Good one-sentence summary

Robotics is AI under uncertainty **plus** geometry **plus** physics **plus** real-time action.



### Hardware categories worth remembering

- **Manipulators**: robot arms for grasping, moving, assembling
- **Mobile robots**: wheeled robots, autonomous cars, rovers, drones
- **Assistive robots**: exoskeletons, prostheses, wheelchair-mounted arms
- **Service robots**: delivery robots, cleaning robots, hospital robots

### Sensor categories worth remembering

- **Exteroceptive sensors**: tell the robot about the environment
  - camera, lidar, radar, sonar, tactile skin
- **Proprioceptive sensors**: tell the robot about itself
  - encoders, gyroscopes, accelerometers, torque sensors

### Actuator categories worth remembering

- electric actuators
- hydraulic actuators
- pneumatic actuators

In AI-oriented robotics, hardware matters because it determines what the robot can perceive, how precisely it can move, and what kinds of algorithms are realistic.



## 2. The robotics problem: perception, planning, control

A clean way to teach robotics is to separate three levels.

### Perception

Estimate what matters from sensor data.

Examples:
- where is the robot?
- where are obstacles?
- where are people?

### Planning

Choose a path, sequence of actions, or policy.

Examples:
- how do I get from room A to room B?
- how do I move the arm to a grasp pose without collision?

### Control

Convert desired motion into low-level actuator commands.

Examples:
- how much torque should each joint apply right now?
- how do I stay close to the planned trajectory despite noise?

---

## Crucial distinction

### Path vs trajectory vs policy

- **Path**: geometric route through space or configuration space
- **Trajectory**: a path **with timing**
- **Policy**: a rule mapping current state to action

This distinction is one of the most important in the chapter.



### Robotics and AI formalisms

Depending on what is known and what is uncertain, robotics problems can be modeled as:

- **Search problems**: if the world is simple and deterministic
- **MDPs**: if the world is stochastic but fully observable
- **POMDPs**: if the robot has uncertainty about the true state
- **Games / multiagent models**: if the robot must coordinate with humans or other robots

In practice, full end-to-end solutions are often too hard. So robotic systems usually decompose the problem into modules.

That decomposition is useful, but it is also imperfect. Better integrated systems are an active research theme.



## Quick check 1

1. Why is a robot not just an ordinary software agent?  
2. What is the difference between perception and control?  
3. Why is a trajectory not the same thing as a path?  
4. Why are continuous state spaces harder than small discrete ones?



## 3. Robotic perception

A robot needs an **internal representation** of the world that is good enough for decision making.

### Three desirable properties of a robot representation

A good representation should:

1. contain enough information for useful decisions;
2. be efficient to update;
3. correspond to natural state variables in the real world.

### Core perception tasks

- **Localization**: where is the robot?
- **Mapping**: what does the environment look like?
- **SLAM**: estimate both at once

### Why probability appears everywhere

Sensors are noisy. Motions are noisy. Therefore the robot often represents belief as a **distribution over possible states**, not a single perfect guess.

Two classical families:

- **Kalman-filter family**: compact Gaussian beliefs, strong assumptions, computationally efficient
- **Particle-filter family**: sample-based beliefs, more flexible, handles multimodal uncertainty better



## Demo 1 — Particle-filter localization in 1D

This demo is intentionally simple. The robot moves in a one-dimensional corridor from 0 to 100.

- The true robot position is hidden from the estimator.
- The robot receives a noisy control command and a noisy distance measurement to the nearest landmark.
- A particle filter maintains many hypotheses about the robot position.

This is **not** industrial robotics code. It is a teaching demo for the core idea of Monte Carlo localization.


In [ ]:

# 1D particle-filter localization demo
world_min, world_max = 0.0, 100.0
landmarks = np.array([15.0, 55.0, 85.0])
T = 18
u = 4.0                  # commanded movement each step
motion_sigma = 1.2       # motion noise
sensor_sigma = 2.5       # sensor noise
num_particles = 600

true_x = 8.0
particles = np.random.uniform(world_min, world_max, size=num_particles)
weights = np.ones(num_particles) / num_particles

true_history = []
estimate_history = []
spread_history = []
particle_snapshots = {}


def nearest_landmark_distance(x):
    return np.min(np.abs(landmarks - x))

for t in range(T):
    # True robot motion
    true_x = np.clip(true_x + u + np.random.normal(0, motion_sigma), world_min, world_max)

    # Noisy sensor reading: distance to nearest landmark
    z = nearest_landmark_distance(true_x) + np.random.normal(0, sensor_sigma)

    # Prediction step
    particles = particles + u + np.random.normal(0, motion_sigma, size=num_particles)
    particles = np.clip(particles, world_min, world_max)

    # Update step
    expected = np.min(np.abs(particles[:, None] - landmarks[None, :]), axis=1)
    likelihood = np.exp(-0.5 * ((z - expected) / sensor_sigma) ** 2)
    likelihood += 1e-12
    weights = likelihood / likelihood.sum()

    # Estimate
    estimate = np.sum(weights * particles)
    spread = np.sqrt(np.sum(weights * (particles - estimate) ** 2))

    true_history.append(true_x)
    estimate_history.append(estimate)
    spread_history.append(spread)

    if t in [0, 4, 9, 17]:
        particle_snapshots[t] = (particles.copy(), weights.copy(), z, true_x, estimate)

    # Resampling
    indices = np.random.choice(np.arange(num_particles), size=num_particles, p=weights)
    particles = particles[indices]
    weights = np.ones(num_particles) / num_particles

# Plot estimate over time
steps = np.arange(1, T + 1)
plt.figure()
plt.plot(steps, true_history, marker='o', label='True position')
plt.plot(steps, estimate_history, marker='s', label='Particle filter estimate')
plt.fill_between(
    steps,
    np.array(estimate_history) - np.array(spread_history),
    np.array(estimate_history) + np.array(spread_history),
    alpha=0.2,
    label='Approx. belief spread'
)
plt.xlabel('Time step')
plt.ylabel('Position in corridor')
plt.title('Particle-filter localization: true position vs estimate')
plt.legend()
plt.show()


In [ ]:

# Visualize selected particle snapshots
snapshots = sorted(particle_snapshots.items())
fig, axes = plt.subplots(len(snapshots), 1, figsize=(9, 10), sharex=True)

if len(snapshots) == 1:
    axes = [axes]

for ax, (t, (p, w, z, tx, est)) in zip(axes, snapshots):
    ax.hist(p, bins=40, weights=np.ones_like(p) / len(p), alpha=0.7)
    for lm in landmarks:
        ax.axvline(lm, linestyle='--', alpha=0.8)
    ax.axvline(tx, linewidth=2, label='True' if t == snapshots[0][0] else None)
    ax.axvline(est, linewidth=2, linestyle=':', label='Estimate' if t == snapshots[0][0] else None)
    ax.set_ylabel(f't={t+1}')
    ax.set_title(f'Measured nearest-landmark distance ≈ {z:.2f}')

axes[-1].set_xlabel('Corridor position')
axes[0].legend()
plt.tight_layout()
plt.show()



### What to notice in Demo 1

- The filter starts with broad uncertainty.
- As motion and measurements accumulate, particles concentrate around plausible positions.
- The estimate is not perfect, but the belief usually becomes sharper.
- Particle filters are attractive when the belief may be **multimodal** or strongly non-Gaussian.

### Why this matters

A robot often cannot act intelligently unless it first has a reasonable estimate of where it is.



## Quick check 2

1. Why does localization require probability rather than a single deterministic position estimate?  
2. What kind of uncertainty is the particle filter representing?  
3. When might a Kalman-filter style Gaussian belief be too restrictive?



## 4. Configuration space and motion planning

### Core idea

Robots do not plan directly over “points in the room.” They often plan over **configurations**.

A configuration specifies everything needed to determine where the robot’s body is.

### Examples

- A point robot in a plane: configuration might be `(x, y)`
- A mobile robot with orientation: `(x, y, θ)`
- A 2-link arm: `(θ₁, θ₂)`

### Why configuration space matters

Collision depends on the **whole robot body**, not just one point.

This is the key conceptual move:
- **workspace** = physical world;
- **configuration space (C-space)** = abstract space of robot poses/joint settings.

Motion planning asks for a route through **free C-space**.



### Four planning styles students should distinguish

#### 1. Visibility graph
Useful in low-dimensional polygonal settings. Often produces very short paths, but they can pass uncomfortably close to obstacles.

#### 2. Cell decomposition / grid search
Discretize space and run search such as A*. Easy to teach, easy to implement, but scales poorly in high dimensions.

#### 3. PRM (Probabilistic Roadmap)
Sample collision-free milestones and connect them. Good for **multi-query** settings.

#### 4. RRT (Rapidly-exploring Random Tree)
Grow a search tree through continuous space. Good for **single-query** planning and high-dimensional spaces.

### Important contrast

- **PRM** builds a reusable graph of the space.
- **RRT** aggressively explores toward new random samples.



## Demo 2 — A* motion planning on a 2D occupancy grid

This demo is deliberately simpler than full continuous robot motion planning.

It still captures several essential ideas:

- obstacles block motion;
- the planner works in a representation of free vs occupied space;
- the output is a **path**, not motor torques;
- planning quality depends on the representation.


In [ ]:

# A* path planning on a grid
rows, cols = 25, 35
grid = np.zeros((rows, cols), dtype=int)

# Build obstacles
grid[4:20, 8] = 1
grid[5, 8:18] = 1
grid[19, 8:26] = 1
grid[8:19, 18] = 1
grid[2:14, 26] = 1
grid[13, 22:33] = 1
grid[20:24, 15:30] = 1

start = (2, 2)
goal = (22, 32)

def heuristic(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def neighbors(node):
    r, c = node
    cand = [(r-1,c), (r+1,c), (r,c-1), (r,c+1)]
    out = []
    for rr, cc in cand:
        if 0 <= rr < rows and 0 <= cc < cols and grid[rr, cc] == 0:
            out.append((rr, cc))
    return out

def astar(start, goal):
    frontier = []
    heapq.heappush(frontier, (0, start))
    came_from = {start: None}
    g_score = {start: 0}

    while frontier:
        _, current = heapq.heappop(frontier)
        if current == goal:
            break

        for nxt in neighbors(current):
            tentative = g_score[current] + 1
            if nxt not in g_score or tentative < g_score[nxt]:
                g_score[nxt] = tentative
                f = tentative + heuristic(nxt, goal)
                heapq.heappush(frontier, (f, nxt))
                came_from[nxt] = current

    if goal not in came_from:
        return None

    path = []
    cur = goal
    while cur is not None:
        path.append(cur)
        cur = came_from[cur]
    path.reverse()
    return path

path = astar(start, goal)
print(f"Path found: {path is not None}. Path length = {len(path) if path else 'N/A'}")


In [ ]:

# Plot the grid and the path
canvas = np.ones((rows, cols, 3))
canvas[grid == 1] = [0.2, 0.2, 0.2]   # obstacles
canvas[grid == 0] = [0.95, 0.95, 0.95]

plt.figure(figsize=(10, 7))
plt.imshow(canvas, origin='upper')

if path is not None:
    rr = [p[0] for p in path]
    cc = [p[1] for p in path]
    plt.plot(cc, rr, linewidth=2, label='A* path')

plt.scatter(start[1], start[0], s=100, marker='o', label='Start')
plt.scatter(goal[1], goal[0], s=100, marker='*', label='Goal')
plt.title('A* path planning on a grid world')
plt.legend()
plt.gca().invert_yaxis()
plt.show()



### What to notice in Demo 2

- The planner works on a **discrete abstraction** of space.
- A* gives a route through free cells.
- This is planning, not control. The result says where to go, not what torque to apply.
- In higher-dimensional robotics, grid methods quickly become expensive due to the **curse of dimensionality**.

### Limitation of the demo

A real robot arm or car lives in a richer space than a 2D occupancy grid. But the demo is still useful because it makes the planning idea concrete.



## Quick check 3

1. Why is motion planning usually done in configuration space rather than only in workspace?  
2. What is the difference between A* on a grid and a continuous-space planner like RRT?  
3. Why can the shortest path be a bad path for a real robot?



## 5. Trajectory tracking and feedback control

Once a planner has produced a path or trajectory, the robot still needs to **move correctly**.

### Core distinction

Planning answers: **What route should I take?**  
Control answers: **What action should I apply right now to stay on track?**

### Why open-loop execution is fragile

Suppose you computed perfect torques from a dynamics model. In reality:

- friction may differ from the model;
- masses may be imperfectly known;
- disturbances may appear;
- sensors may be noisy.

Therefore, robots need **feedback**.

### Classical controllers

#### P control
Correct proportionally to the position error.

#### PD control
Add a derivative term that damps oscillation.

#### PID control
Add an integral term to eliminate persistent bias / steady-state error.

For a first course, the main thing to retain is:

- P may overshoot;
- PD is often much better behaved;
- PID helps when systematic error remains.



## Demo 3 — P vs PD control on a simple mass

We simulate a 1D point mass that should move from position 0 to position 1.

The dynamics are intentionally simple:

- state = position and velocity
- action = force
- we compare a **P controller** and a **PD controller**

This is a minimal illustration of why derivative feedback matters.


In [ ]:

# P vs PD control demo for a 1D point mass
T = 8.0
dt = 0.01
steps = int(T / dt)
time = np.arange(steps) * dt
target = 1.0


def simulate(controller='P', kp=14.0, kd=0.0):
    x = 0.0
    v = 0.0
    x_hist, v_hist, u_hist = [], [], []
    for _ in time:
        error = target - x
        derror = -v
        if controller == 'P':
            u = kp * error
        elif controller == 'PD':
            u = kp * error + kd * derror
        else:
            raise ValueError('Unknown controller')

        # Simple mass with mild friction
        a = u - 0.35 * v
        v = v + a * dt
        x = x + v * dt

        x_hist.append(x)
        v_hist.append(v)
        u_hist.append(u)
    return np.array(x_hist), np.array(v_hist), np.array(u_hist)

x_p, v_p, u_p = simulate('P', kp=14.0, kd=0.0)
x_pd, v_pd, u_pd = simulate('PD', kp=14.0, kd=6.5)

plt.figure()
plt.plot(time, x_p, label='P controller')
plt.plot(time, x_pd, label='PD controller')
plt.axhline(target, linestyle='--', label='Target')
plt.xlabel('Time (s)')
plt.ylabel('Position')
plt.title('Position tracking: P vs PD')
plt.legend()
plt.show()


In [ ]:

plt.figure()
plt.plot(time, u_p, label='P controller')
plt.plot(time, u_pd, label='PD controller')
plt.xlabel('Time (s)')
plt.ylabel('Control input')
plt.title('Control effort: P vs PD')
plt.legend()
plt.show()



### What to notice in Demo 3

- The P controller often oscillates more.
- The PD controller damps the motion and usually settles faster.
- The derivative term reacts to how fast the error is changing, not only to how large it is.

### Conceptual takeaway

Motion planning without feedback is not enough. Real robots need control laws that compensate for error during execution.



## 6. Uncertainty, replanning, and information gathering

Even after we separate perception, planning, and control, real robots still face uncertainty.

### Two important ideas

#### Online replanning / Model Predictive Control (MPC)
Plan, execute a little, observe again, replan.

This is practical because the robot does not trust its plan forever.

#### Information-gathering actions
Sometimes the robot should act not only to make progress, but to **reduce uncertainty**.

Examples:
- move closer to a landmark to localize better;
- probe until contact is sensed;
- adjust viewpoint to see an object more clearly.

### Deep lesson

The best action is not always the one that makes immediate progress. Sometimes it is the one that improves future knowledge.



## 7. Reinforcement learning in robotics

Robotics and reinforcement learning fit naturally together, because robots often do not have a perfect dynamics model.

### Why RL in robotics is attractive

- robots can learn skills from trial and error;
- policies can be learned directly from experience;
- high-dimensional perception can be integrated with action.

### Why RL in robotics is difficult

- real-world data is slow to collect;
- unsafe exploration is unacceptable;
- sample complexity is a major bottleneck;
- simulation may not perfectly match reality.

### Terms worth remembering

- **model-based RL**: use or learn dynamics models
- **sim-to-real transfer**: train in simulation, transfer to hardware
- **domain randomization**: vary simulation parameters so the learned policy is robust
- **motion primitives**: higher-level skills that reduce the learning burden

For a first pass, the most important point is this: robotics RL is shaped by the cost of real data and the need for safety.



## 8. Humans and robots

Robots are often built to work **with** humans, not away from them.

This introduces two major challenges.

### A. Coordination

The robot must act in ways that mesh well with human behavior.

Examples:
- autonomous car merging with human drivers;
- delivery robot moving through a crowded corridor;
- assistant robot handing tools to a person.

### B. Learning what humans actually want

A robot may not know the correct reward function ahead of time.

Two major approaches:

#### Preference learning
Infer the objective or cost function from demonstrations, corrections, or comparisons.

#### Imitation learning
Learn the policy directly from demonstrated behavior.

### Important contrast

- **Preference learning** asks: *what objective explains the behavior?*
- **Imitation learning** asks: *what action should I copy in this state?*

Imitation can work well, but it can also fail badly when the robot drifts into states that were not represented in the demonstrations.



## 9. Reactive robotics as an alternative viewpoint

Not all robotics systems explicitly build rich world models and deliberate over long plans.

### Reactive view

A reactive controller maps perception more directly to action.

Examples:
- obstacle avoidance reflexes
- wall following
- simple legged locomotion rules

### Why reactive approaches matter

They can be:
- simpler,
- faster,
- more robust in some tightly constrained settings.

### Why they are limited

They usually struggle with:
- long-horizon reasoning,
- explicit goal changes,
- rich uncertainty representation,
- complex human interaction.

This is why modern robotics often combines reactive and deliberative components rather than choosing only one.



## Exercises

### Conceptual exercises

1. Explain in your own words why robotics is often modeled with POMDP-like ideas rather than plain deterministic search.
2. Give one example where localization uncertainty could lead to a dangerous robot action.
3. Explain why a path planner alone does not solve the robot control problem.
4. Compare PRM and RRT in two or three sentences.
5. Explain one failure mode of imitation learning.
6. Give one example of an information-gathering action for a household robot.

### Short applied exercises

1. Modify the particle-filter demo by increasing `sensor_sigma`. What happens to the estimate quality?
2. Add more obstacles to the A* grid. Can you create a case where no path exists?
3. Change the gains in the control demo. Which values make the P controller oscillate more strongly?
4. Try changing the damping term in the simulated dynamics. How does that affect the PD controller?



## Summary: what to remember from Chapter 26

If you remember only a few ideas, remember these:

1. **Robotics is embodied AI**: sensing, reasoning, and acting in the physical world.
2. **Perception, planning, and control are different problems** and should not be conflated.
3. **Uncertainty is fundamental**: robots usually need probabilistic estimation.
4. **Configuration space is central** to motion planning.
5. **Feedback control is essential** because real execution deviates from ideal plans.
6. **Learning in robotics is constrained by real-world data, safety, and sim-to-real issues.**
7. **Humans matter twice**: robots must coordinate with us and also learn what we want.



## Optional further reading

For students who want to go deeper after this notebook:

- *Artificial Intelligence: A Modern Approach* — Chapter 26
- Thrun, Burgard, Fox — *Probabilistic Robotics*
- LaValle — *Planning Algorithms*
- Lynch and Park — *Modern Robotics*
- Siciliano and Khatib — *Springer Handbook of Robotics*

A good next technical step after this notebook would be one of these:

- implement 2D localization in a richer environment;
- implement PRM or RRT in continuous space;
- simulate PID control more fully;
- build a tiny imitation-learning experiment.
